### Import libraries and files

In [16]:
from pathlib import Path
import pandas as pd
import json
import requests
import re

WORKFLOW = Path.cwd().parent
DATA = WORKFLOW / "data"

master_file = pd.read_csv(DATA / "sheet1.csv", skiprows=2, low_memory=False)

genome_loc = pd.read_csv(DATA / "sheet2.csv")

master_file.loc[master_file['effect'] == 'LoF', 'effect'] = 'loss_of_function_variant'
master_file.loc[master_file['effect'] == 'frameshift', 'effect'] = 'frameshift_variant'

gff = pd.read_csv("/data/scratch/projects/punim1703/tbvcf/resources/NC_000962.3.gff3", sep="\t", comment="#", header=None)
gff.columns = ["seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"]

gff["locus_tag"] = 'null'

gff["name"] = 'null' 

for index, row in gff.iterrows():
    id = re.search(r"ID=[^-]+-(\w+)", row["attributes"])
    name = re.search(r";Name=([^;]+)", row["attributes"])

    gff.loc[index,'locus_tag'] = id.group(1) if id else 'null'
    gff.loc[index,'name'] = name.group(1).lower() if name else 'null'

gff['strand']

0       +
1       +
2       +
3       +
4       +
       ..
8402    -
8403    -
8404    -
8405    -
8406    -
Name: strand, Length: 8407, dtype: object

### Only keep the columns of interest

In [17]:
master_file = pd.merge(master_file, genome_loc, on="variant")

master_file = master_file[
    [
           "variant",
           "gene",
           "effect",
           "drug",
           "FINAL CONFIDENCE GRADING"
    ]
]


### Strip away other text in the final confidence grading

In [27]:
for index, row in master_file.iterrows():
    master_file.at[index,"FINAL CONFIDENCE GRADING"] = row["FINAL CONFIDENCE GRADING"][0]

master_file = master_file.loc[
    (master_file["FINAL CONFIDENCE GRADING"] == "1")
    | (master_file["FINAL CONFIDENCE GRADING"] == "2")
]

master_file[master_file['gene'] == 'ethA']

,variant,gene,effect,drug,FINAL CONFIDENCE GRADING
91562,ethA_c.-7T>C,ethA,upstream_gene_variant,Ethionamide,2
91586,ethA_LoF,ethA,loss_of_function_variant,Ethionamide,1
91587,ethA_LoF,ethA,loss_of_function_variant,Ethionamide,1
91588,ethA_LoF,ethA,loss_of_function_variant,Ethionamide,1
91589,ethA_LoF,ethA,loss_of_function_variant,Ethionamide,1
...,...,...,...,...,...
96900,ethA_p.Val489fs,ethA,frameshift_variant,Ethionamide,2
96901,ethA_p.Val489fs,ethA,frameshift_variant,Ethionamide,2
96902,ethA_p.Val489fs,ethA,frameshift_variant,Ethionamide,2
96914,ethA_p.Val85fs,ethA,frameshift_variant,Ethionamide,2


In [50]:
# All genes mentioned
genes = list(master_file['gene'].unique())

gene_dict = {}

for gene in genes:
    subset = master_file[master_file['gene'] == gene]
    info = gff[gff['name'] == gene.lower()]

    locus = info["locus_tag"].iloc[0]
    start = int(info["start"].iloc[0]) if info['strand'].iloc[0] == '+' else int(info["end"].iloc[0])
    end = int(info["end"].iloc[0]) if info['strand'].iloc[0] == '+' else int(info["start"].iloc[0])
    
    gene_dict.update({
            gene: {
                "gene_id": locus,
                "Region": {
                    "start": start,
                    "end": end
                },
                "Rules": {}
        }})

    # If not upstream variant         
    for index, row in subset.iterrows():
        if row['effect'] == 'upstream_gene_variant':
            continue

        if row['effect'] not in gene_dict[gene]["Rules"]:
            gene_dict[gene]["Rules"][row['effect']] = {}

        gene_dict[gene]["Rules"][row['effect']][row['drug']] = row['FINAL CONFIDENCE GRADING']

    if 'upstream_gene_variant' in subset['effect'].values:
        gene_dict.update({
            f'{gene}_upstream': {
                "gene_id": locus,
                "Region": {
                    "start": start,
                    "end": end
                },
                "Rules": {}
        }})
        up_set = subset[subset['effect']=='upstream_gene_variant']

        if 'upstream_gene_variant' not in gene_dict[f'{gene}_upstream']["Rules"]:
            gene_dict[f'{gene}_upstream']["Rules"]['upstream_gene_variant'] = {}
            for i, row in up_set.iterrows():
                gene_dict[f'{gene}_upstream']["Rules"]['upstream_gene_variant'][row['drug']] = row['FINAL CONFIDENCE GRADING']

        if len(subset['effect'].unique()) == 1:
            continue

    


gene_dict



{'eis': {'gene_id': 'Rv2416c',
  'Region': {'start': 2715332, 'end': 2714124},
  'Rules': {}},
 'eis_upstream': {'gene_id': 'Rv2416c',
  'Region': {'start': 2715332, 'end': 2714124},
  'Rules': {'upstream_gene_variant': {'Amikacin': '2', 'Kanamycin': '2'}}},
 'rrs': {'gene_id': 'Rvnr01',
  'Region': {'start': 1471846, 'end': 1473382},
  'Rules': {'non_coding_transcript_exon_variant': {'Amikacin': '1',
    'Capreomycin': '1',
    'Kanamycin': '1',
    'Streptomycin': '2'}}},
 'atpE': {'gene_id': 'Rv1305',
  'Region': {'start': 1461045, 'end': 1461290},
  'Rules': {'missense_variant': {'Bedaquiline': '2'}}},
 'pepQ': {'gene_id': 'Rv2535c',
  'Region': {'start': 2860418, 'end': 2859300},
  'Rules': {'loss_of_function_variant': {'Bedaquiline': '2',
    'Clofazimine': '2'},
   'frameshift_variant': {'Bedaquiline': '2', 'Clofazimine': '2'},
   'stop_gained': {'Bedaquiline': '2'}}},
 'Rv0678': {'gene_id': 'Rv0678',
  'Region': {'start': 778990, 'end': 779487},
  'Rules': {'loss_of_function_va

### Output the JSON

In [52]:
json_str = json.dumps(gene_dict)

with open(WORKFLOW/"results/ER.json", "w") as f:
    f.write(json_str)